# 08 — Multimodal Retrieval-Augmented Generation

## What
Multimodal RAG combines a cross-modal retrieval system (typically CLIP-based) with a VLM generator. Given a query (text or image), the system retrieves relevant images, documents, or image-text pairs from a knowledge base, then conditions the generator on both the query and retrieved context.

## Why
VLMs are limited by what they memorised during training. Multimodal RAG lets models access up-to-date information, proprietary image databases, or specialised visual knowledge without retraining. It is the multimodal equivalent of text RAG and is increasingly used in medical imaging, e-commerce visual search, and document intelligence.

## Community context
Wikipedia-based multimodal RAG (RAVEN, Ra-CM3) established the research paradigm. Production systems (Google Lens, Pinterest visual search) use approximate nearest-neighbour (ANN) search via FAISS, ScaNN, or Weaviate over CLIP embedding indices.

In [ ]:
# Multimodal retrieval: CLIP-based cross-modal nearest neighbour search
import numpy as np

class MultimodalVectorStore:
    """
    Simple CLIP-based vector store for cross-modal retrieval.
    Stores (image_embed, metadata) pairs, retrieves by cosine similarity.
    """
    def __init__(self, embed_dim=128):
        self.embeds = []   # image embeddings
        self.metas  = []   # associated metadata

    def add(self, image_embed, metadata):
        norm_embed = image_embed / (np.linalg.norm(image_embed) + 1e-8)
        self.embeds.append(norm_embed)
        self.metas.append(metadata)

    def retrieve(self, query_embed, k=3):
        """Retrieve top-k items by cosine similarity to query"""
        q = query_embed / (np.linalg.norm(query_embed) + 1e-8)
        scores = [np.dot(q, e) for e in self.embeds]
        ranked = sorted(enumerate(scores), key=lambda x: -x[1])
        return [(self.metas[i], round(s,4)) for i, s in ranked[:k]]

class MultimodalRAG:
    def __init__(self, vector_store):
        self.store = vector_store

    def query(self, query_embed, query_text, k=3):
        """Retrieve relevant images then 'generate' a response"""
        retrieved = self.store.retrieve(query_embed, k)
        print(f'Query: "{query_text}"')
        print(f'\nTop-{k} retrieved items:')
        for meta, score in retrieved:
            print(f'  [{score:.4f}] {meta["title"]} | {meta["description"]}')
        context = ' | '.join([m['description'] for m, _ in retrieved])
        response = f'Based on retrieved images ({context}): [VLM generated response to "{query_text}"]'
        return response

np.random.seed(42)
D = 128

# Build knowledge base of medical imaging examples
store = MultimodalVectorStore(D)
knowledge_base = [
    {'title': 'Normal chest X-ray',    'description': 'clear lungs, normal heart size', 'domain': 'radiology'},
    {'title': 'Pneumonia chest X-ray', 'description': 'bilateral infiltrates, air space disease', 'domain': 'radiology'},
    {'title': 'Pleural effusion',      'description': 'blunting of costophrenic angle', 'domain': 'radiology'},
    {'title': 'Lung nodule CT',        'description': 'solid nodule right upper lobe 8mm', 'domain': 'radiology'},
    {'title': 'Normal ECG',            'description': 'sinus rhythm 72bpm no ST changes', 'domain': 'cardiology'},
    {'title': 'STEMI ECG',             'description': 'ST elevation V1-V4 anterior MI', 'domain': 'cardiology'},
]

# Create correlated embeddings: similar images are close
base_radiology = np.random.randn(D)
base_cardiology = np.random.randn(D)
embeddings = [
    base_radiology + np.random.randn(D)*0.1,
    base_radiology + np.random.randn(D)*0.3,
    base_radiology + np.random.randn(D)*0.3,
    base_radiology + np.random.randn(D)*0.4,
    base_cardiology + np.random.randn(D)*0.1,
    base_cardiology + np.random.randn(D)*0.2,
]
for emb, meta in zip(embeddings, knowledge_base):
    store.add(emb, meta)

rag = MultimodalRAG(store)

# Query: image of an ECG (close to cardiology embeddings)
ecg_query = base_cardiology + np.random.randn(D)*0.15
resp = rag.query(ecg_query, 'What abnormality is visible in this ECG?', k=3)
print(f'\nGenerated response:\n  {resp}')

## Re-ranking with Multimodal Fusion

First-stage retrieval uses fast approximate search. Second-stage re-ranking uses a more expensive cross-modal model that jointly encodes query + retrieved item for more accurate relevance scoring.

In [ ]:
# Cross-modal re-ranker: jointly scores query text + image embed
def cross_modal_rerank(query_text_embed, retrieved_image_embeds, alpha=0.6):
    """
    Fuse text and image similarity for re-ranking.
    alpha controls text vs image weight.
    """
    q_text = query_text_embed / (np.linalg.norm(query_text_embed) + 1e-8)
    scores = []
    for img_emb in retrieved_image_embeds:
        q_img = ecg_query / (np.linalg.norm(ecg_query) + 1e-8)
        img   = img_emb   / (np.linalg.norm(img_emb)   + 1e-8)
        img_score  = np.dot(q_img,   img)
        text_score = np.dot(q_text,  img)  # use image emb as proxy for text similarity
        fused = alpha * img_score + (1-alpha) * text_score
        scores.append(fused)
    return scores

query_text_emb = base_cardiology + np.random.randn(D)*0.1  # 'ECG abnormality'
retrieved_embs = embeddings  # all 6 for demo
rerank_scores = cross_modal_rerank(query_text_emb, retrieved_embs)
print('Re-ranked results (text+image fusion):')
ranked = sorted(zip(knowledge_base, rerank_scores), key=lambda x: -x[1])
for meta, score in ranked:
    print(f'  [{score:.4f}] {meta["title"]}')